In [ ]:
import cv2
import numpy as np
import time, csv, datetime, os
import ctypes

# ==========================================
# USER CONFIGURATION
# ==========================================
ID_CAM = 0

# --- INITIAL RECTANGLE (centre x/y, width, height in thermal pixels) ---
RECT_CONFIG = {'cx': 115, 'cy': 121, 'w': 29, 'h': 27}

# --- GEOMETRY SETTINGS ---
ANGLE_CORRECTION = {'x': 1.0, 'y': 1.0, 'taper': 1.0}

# --- Calibration ---
OFFSET    = 0.0      # °C offset added to every pixel
EMISSIVITY = 0.95
SCALE      = 3       # display upscale factor

# --- POLLING RATE ---
# Minimum time (seconds) between frame reads from the camera.
# The camera hardware is fixed at ~25 fps, so values below 0.04 have no effect.
# Increase this to reduce CPU load or to record at a lower effective rate.
#   0.04  → ~25 fps  (camera maximum)
#   0.1   → ~10 fps
#   1.0   →   1 fps
POLL_INTERVAL = 0.4   # seconds

# --- RESOLUTION ---
# Fraction of pixels to sample from the ROI (applied uniformly in x and y).
#   1.0  → every pixel  (full resolution)
#   0.5  → every 2nd pixel in each direction (~25% of pixels)
#   0.25 → every 4th pixel (~6% of pixels)
# Step size is rounded to the nearest integer, minimum 1.
RESOLUTION_COEFF = 1.0

# --- AUTO-STOP TIMER ---
# When set to a number of seconds, both CSV and video auto-stop and save
# when the timer expires.  [T] / [S] always work regardless and will also
# cancel the remaining timer for that stream.
# Set to None to disable the timer entirely.
RECORD_DURATION = 22000   # e.g.  30  to auto-stop after 30 s

# --- CSV STREAMING ---
# Rows are written to disk every N frames rather than held in RAM.
# Lower values = less data at risk if the kernel crashes; higher = fewer disk writes.
CSV_FLUSH_INTERVAL = 500   # flush to disk every N recorded frames

# --- NOZZLE TRACKER ---
# TRACKER_CENTER_X  : nominal horizontal centre (thermal pixels from left)
# TRACKER_X_BAND    : half-width of X search band; allows gentle X drift compensation.
# TRACKER_BOX_W/H   : half-size of the sampled box (CSV pixels + display box).
# TRACKER_Y_SMOOTH  : EMA for Y (height). 0.25 = moderate, 1.0 = raw.
# TRACKER_X_SMOOTH  : EMA for X (lateral). Keep low - only for gentle drift.
# TRACKER_SNAP_THR  : Snap directly if detected Y differs by more than this many rows.
#                     Handles fast travel moves without chasing slowly.
# TRACKER_MIN_CONF  : Min peak temp (C) in search band for valid nozzle lock.
#                     Below this the box turns grey (nozzle not visible).
ENABLE_TRACKER    = True
TRACKER_CENTER_X  = 160   # nominal horizontal centre in thermal pixels
TRACKER_X_BAND    = 0    # half-width of X search band (thermal pixels)
TRACKER_BOX_W     = 58     # half-width of sampled box (thermal pixels)
TRACKER_BOX_H     = 30     # half-height of sampled box (thermal pixels)
TRACKER_Y_SMOOTH  = 0.25  # EMA for Y: 1.0 = raw/snappy, 0.1 = heavy
TRACKER_X_SMOOTH  = 0.05  # EMA for X: keep small, just for gentle drift
TRACKER_SNAP_THR  = 8     # rows - snap instead of EMA on large jumps
TRACKER_MIN_CONF  = 50.0  # C - below this = low confidence (grey box)
# Prediction-gated search radius (rows). When the tracker has a lock,
# it first searches only within this many rows of the current position.
# A reflection outside this window cannot steal the lock, even if hotter.
# Only falls back to the full band if nothing above TRACKER_MIN_CONF is
# found in the gated window (nozzle lost / print not started).
# Set to ~3x the maximum genuine nozzle movement per frame.
# At 25fps and typical delta speeds: 5-10 rows is usually enough.
TRACKER_GATE_RADIUS = 8    # rows around current position to search first
# Gaussian blur radius applied to the search band before finding the peak.
# Suppresses single-pixel reflections (which a small blur destroys) while
# preserving the nozzle blob (which is spatially extended).
# 0 = disabled, 1.5 = good default, higher = more suppression but more lag.
TRACKER_BLOB_SIGMA = 1.5


# ==========================================
# HELPER FUNCTIONS
# ==========================================
def is_shift_pressed():
    try:
        return (ctypes.windll.user32.GetKeyState(0x10) & 0x8000) != 0
    except AttributeError:      # non-Windows
        return False

def get_rect_bounds(cfg):
    """Return (x1, y1, x2, y2) integer pixel bounds from centre/size config."""
    x1 = cfg['cx'] - cfg['w'] // 2
    y1 = cfg['cy'] - cfg['h'] // 2
    x2 = x1 + cfg['w']
    y2 = y1 + cfg['h']
    return x1, y1, x2, y2

def get_base_corners(cfg):
    x1, y1, x2, y2 = get_rect_bounds(cfg)
    return [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]

def decode_temperatures(thermal: np.ndarray) -> np.ndarray:
    lo  = thermal[..., 0].astype(np.int32)
    hi  = thermal[..., 1].astype(np.int32)
    raw = lo + (hi << 8)
    return (raw / 64.0) - 273.15

def extract_roi_pixels(temps, cfg, taper=1.0, step=1):
    """
    Return sampled temperatures from the (possibly tapered) ROI.
    `step` controls sub-sampling: 1 = every pixel, 2 = every 2nd, etc.
    Pixels are visited row-by-row, left-to-right at the given step.
    Out-of-bounds pixels are nan.
    """
    x1, y1, x2, y2 = get_rect_bounds(cfg)
    center_x   = (x1 + x2) / 2.0
    rows_total = y2 - y1

    values = []
    for iy, py in enumerate(range(y1, y2, step)):
        v = iy * step / (rows_total - 1) if rows_total > 1 else 0.5
        v = min(v, 1.0)
        row_scale = 1.0 + (taper - 1.0) * v
        for px in range(x1, x2, step):
            dist = px - center_x
            tx   = int(round(center_x + dist * row_scale))
            if 0 <= py < temps.shape[0] and 0 <= tx < temps.shape[1]:
                values.append(temps[py, tx])
            else:
                values.append(np.nan)
    return values

def open_camera(index):
    cap = cv2.VideoCapture(index, cv2.CAP_MSMF)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  256)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 384)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('Y','U','Y','2'))
    cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
    return cap

def process_frame(cap, cfg, angle_corr, offset, tracker_pos=None, tracker_confident=True):
    ret, frame = cap.read()
    if not ret:
        return None, None, "FAIL"

    buf = frame.ravel()
    if buf.size != 384 * 256 * 2:
        return None, None, "SKIP"

    reshaped = buf.reshape(384, 256, 2)
    visual_half, thermal_half = np.array_split(reshaped, 2)

    temps = decode_temperatures(thermal_half)
    temps = temps / (EMISSIVITY ** 0.25)
    temps += offset
    temps = np.rot90(temps, 2)

    visual_bgr  = cv2.cvtColor(visual_half, cv2.COLOR_YUV2BGR_YUYV)
    disp        = cv2.resize(visual_bgr, (visual_bgr.shape[1] * SCALE, visual_bgr.shape[0] * SCALE))
    disp_color  = cv2.applyColorMap(disp, cv2.COLORMAP_INFERNO)
    disp_color  = cv2.rotate(disp_color, cv2.ROTATE_180)

    H, W       = disp_color.shape[:2]
    scale_x    = W / temps.shape[1]
    scale_y    = H / temps.shape[0]

    # --- Nozzle tracker box ---
    if ENABLE_TRACKER and tracker_pos is not None:
        tx = int(round(tracker_pos[0] * scale_x))
        ty = int(round(tracker_pos[1] * scale_y))
        bw = int(round(TRACKER_BOX_W * scale_x))
        bh = int(round(TRACKER_BOX_H * scale_y))
        pt1 = (max(0, tx - bw), max(0, ty - bh))
        pt2 = (min(W - 1, tx + bw), min(H - 1, ty + bh))
        # Colour: gold = confident lock, grey = low confidence (nozzle not visible)
        box_col  = (255, 220, 0) if tracker_confident else (160, 160, 160)
        cv2.rectangle(disp_color, pt1, pt2, (255, 255, 255), 3)  # white outline
        cv2.rectangle(disp_color, pt1, pt2, box_col, 1)
        cross_len = max(6, bw // 2)
        cv2.line(disp_color, (tx - cross_len, ty), (tx + cross_len, ty), box_col, 1)
        cv2.line(disp_color, (tx, ty - cross_len), (tx, ty + cross_len), box_col, 1)
        # Nozzle temp = max of the box pixels (not the EMA centre pixel)
        tcx_t = int(round(tracker_pos[0]))
        tcy_t = int(round(tracker_pos[1]))
        ry1 = max(0, tcy_t - TRACKER_BOX_H); ry2 = min(temps.shape[0], tcy_t + TRACKER_BOX_H + 1)
        rx1 = max(0, tcx_t - TRACKER_BOX_W); rx2 = min(temps.shape[1], tcx_t + TRACKER_BOX_W + 1)
        box_region = temps[ry1:ry2, rx1:rx2]
        hot_val  = float(np.nanmax(box_region)) if box_region.size > 0 else float('nan')
        # Max temp in the rows BELOW the nozzle (dr > 0 = recently deposited)
        below_r1 = min(temps.shape[0], tcy_t + 1)
        below_r2 = min(temps.shape[0], tcy_t + TRACKER_BOX_H + 1)
        below_region = temps[below_r1:below_r2, rx1:rx2]
        below_val = float(np.nanmax(below_region)) if below_region.size > 0 else float('nan')
        conf_str = '' if tracker_confident else ' (low conf)'
        lx = min(tx + bw + 4, W - 140)
        ly = max(ty - bh - 6, 16)
        label1 = f"Nozzle: {hot_val:.1f}C{conf_str}"
        label2 = f"Below:  {below_val:.1f}C"
        line_h = 18
        for label, dy in [(label1, 0), (label2, line_h)]:
            cv2.putText(disp_color, label, (lx, ly + dy),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            cv2.putText(disp_color, label, (lx, ly + dy),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_col, 1)

    # --- Overlay info ---
    step      = max(1, round(1.0 / RESOLUTION_COEFF)) if RESOLUTION_COEFF < 1.0 else 1
    roi_vals  = extract_roi_pixels(temps, cfg, taper=angle_corr.get('taper', 1.0), step=step)
    finite    = [v for v in roi_vals if np.isfinite(v)]
    avg_temp  = np.mean(finite) if finite else 0.0

    n_sampled = len(roi_vals)
    info_str  = (f"Off:{offset:+.1f}C "
                 f"| Avg:{avg_temp:.1f}C | Px:{n_sampled} (step={step})")
    cv2.putText(disp_color, info_str, (10, H - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)

    return disp_color, temps, "OK"


# ==========================================
# MAIN LOOP
# ==========================================
def main():
    global RECT_CONFIG, ANGLE_CORRECTION, OFFSET

    recording      = False
    csv_rows       = []      # small write buffer, flushed every CSV_FLUSH_INTERVAL frames
    csv_file       = None    # open file handle while recording
    csv_writer     = None    # csv.writer bound to csv_file
    csv_frame_count = 0      # total frames written so far
    last_csv_time  = 0
    csv_interval   = POLL_INTERVAL     # write one CSV row per polled frame
    video_writer   = None
    record_video   = False
    csv_start      = None          # time.time() when CSV recording began
    vid_start      = None          # time.time() when video recording began
    tracker_pos        = None   # (cx, cy) smoothed nozzle position in thermal pixels
    tracker_confident  = False  # True when peak temp exceeds TRACKER_MIN_CONF
    csv_path       = None          # path of the currently open CSV file

    output_folder  = "Data Recordings"
    os.makedirs(output_folder, exist_ok=True)

    cap = open_camera(ID_CAM)
    if not cap.isOpened():
        print("ERROR: Could not open camera."); return

    timer_info = (f"  Auto-stop after {RECORD_DURATION}s"
                  if RECORD_DURATION else "  Manual stop [T]/[S]")
    print(f"System Started.{timer_info}")
    print("Controls: [Q]uit | [R]ecord CSV | [V]Video | [T]Save CSV | [S]Save Video")
    print("          Arrows: move ROI | Shift+Arrows: resize | [+/-]: offset | [9/0]: taper")

    step = max(1, round(1.0 / RESOLUTION_COEFF)) if RESOLUTION_COEFF < 1.0 else 1
    print(f"  Resolution coeff={RESOLUTION_COEFF} → pixel step={step}")
    print(f"  Poll interval={POLL_INTERVAL}s (~{1/POLL_INTERVAL:.1f} fps max)")

    last_poll = 0.0   # time of last successful frame read

    try:
        while True:
            # ---- Poll-rate throttle ----
            now_poll = time.time()
            if now_poll - last_poll < POLL_INTERVAL:
                time.sleep(0.001)  # yield CPU; keys queue up for the handler below
                continue
            last_poll = now_poll

            # ---- Frame ----
            view, temps, status = process_frame(
                cap, RECT_CONFIG, ANGLE_CORRECTION, OFFSET, tracker_pos, tracker_confident)

            if status == "SKIP": continue
            if status == "FAIL": break

            # ---- Update tracker (prediction-gated blob search) ----
            if ENABLE_TRACKER and temps is not None:
                x1 = max(0, TRACKER_CENTER_X - TRACKER_X_BAND)
                x2 = min(temps.shape[1], TRACKER_CENTER_X + TRACKER_X_BAND + 1)

                def _best_blob(y_lo, y_hi):
                    y_lo = max(0, y_lo); y_hi = min(temps.shape[0], y_hi)
                    if y_lo >= y_hi: return None
                    patch = temps[y_lo:y_hi, x1:x2].copy()
                    patch_min = float(np.nanmin(patch)) if np.isfinite(patch).any() else 0.0
                    patch = np.where(np.isfinite(patch), patch, patch_min)
                    if TRACKER_BLOB_SIGMA > 0:
                        ksize = int(TRACKER_BLOB_SIGMA * 6) | 1
                        blurred = cv2.GaussianBlur(patch.astype(np.float32),
                                                   (ksize, ksize), TRACKER_BLOB_SIGMA)
                    else:
                        blurred = patch
                    flat_idx  = int(np.argmax(blurred))
                    row_local = flat_idx // blurred.shape[1]
                    col_local = flat_idx %  blurred.shape[1]
                    peak_raw  = float(temps[y_lo + row_local, x1 + col_local])
                    return x1 + col_local, y_lo + row_local, peak_raw

                if tracker_pos is None:
                    result = _best_blob(0, temps.shape[0])
                    if result:
                        hx, hy, peak_raw = result
                        tracker_confident = peak_raw >= TRACKER_MIN_CONF
                        tracker_pos = (float(hx), float(hy))
                else:
                    cur_y = tracker_pos[1]
                    # First search only within gate around current position
                    result = _best_blob(int(cur_y) - TRACKER_GATE_RADIUS,
                                        int(cur_y) + TRACKER_GATE_RADIUS + 1)
                    if result and result[2] >= TRACKER_MIN_CONF:
                        hx, hy, peak_raw = result
                        tracker_confident = True
                    else:
                        # Nothing hot near current pos — fall back to full band
                        result = _best_blob(0, temps.shape[0])
                        if result:
                            hx, hy, peak_raw = result
                            tracker_confident = peak_raw >= TRACKER_MIN_CONF
                        else:
                            hx, hy = int(tracker_pos[0]), int(tracker_pos[1])
                            tracker_confident = False
                    dy = abs(hy - cur_y)
                    new_y = float(hy) if dy > TRACKER_SNAP_THR \
                            else TRACKER_Y_SMOOTH * hy + (1 - TRACKER_Y_SMOOTH) * cur_y
                    new_x = TRACKER_X_SMOOTH * hx + (1 - TRACKER_X_SMOOTH) * tracker_pos[0]
                    tracker_pos = (new_x, new_y)

            # ---- Auto-stop timer (per stream, independent) ----
            csv_remaining = None
            vid_remaining = None
            if RECORD_DURATION:
                if recording and csv_start is not None:
                    csv_remaining = RECORD_DURATION - (time.time() - csv_start)
                    if csv_remaining <= 0:
                        recording = False; csv_start = None
                        if csv_rows: csv_writer.writerows(csv_rows); csv_rows = []
                        csv_file.close(); csv_file = None; csv_writer = None
                        print(f'Auto-stop: CSV saved → {csv_path}  ({csv_frame_count} frames)')
                if record_video and vid_start is not None:
                    vid_remaining = RECORD_DURATION - (time.time() - vid_start)
                    if vid_remaining <= 0:
                        record_video = False; vid_start = None
                        if video_writer: video_writer.release(); video_writer = None
                        print("Auto-stop: video saved.")

            # ---- UI bar ----
            ui_h  = 100
            ui_bar = np.zeros((ui_h, view.shape[1], 3), dtype=np.uint8)

            rec_col = (0, 0, 255) if recording    else (80, 80, 80)
            vid_col = (0, 0, 255) if record_video else (80, 80, 80)
            rec_label = f"CSV REC  {csv_frame_count}fr" if recording else "CSV RECORDING"
            cv2.putText(ui_bar, rec_label,        (20, 30),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, rec_col, 2)
            cv2.putText(ui_bar, "VIDEO RECORDING", (280, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, vid_col, 2)

            # Tracker confidence indicator
            if ENABLE_TRACKER:
                conf_col   = (0, 220, 80) if tracker_confident else (80, 80, 160)
                conf_label = "NOZZLE LOCKED" if tracker_confident else "NOZZLE: low conf"
                cv2.putText(ui_bar, conf_label, (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.45, conf_col, 1)

            ts_display = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
            cv2.putText(ui_bar, ts_display,
                        (ui_bar.shape[1] - 220, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # Timer countdowns
            if csv_remaining is not None and csv_remaining > 0:
                cv2.putText(ui_bar, f"CSV stop in {csv_remaining:.1f}s",
                            (280, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 165, 255), 1)
            if vid_remaining is not None and vid_remaining > 0:
                cv2.putText(ui_bar, f"VID stop in {vid_remaining:.1f}s",
                            (280, 68), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 165, 255), 1)

            cmd1 = "[R]CSV  [V]Video  [T]SaveCSV  [S]SaveVid  [Q]uit"
            cmd2 = "[+/-]:Offset"
            cv2.putText(ui_bar, cmd1, (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
            cv2.putText(ui_bar, cmd2, (20, 92), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)

            final = np.vstack([ui_bar, view])

            # Blinking rec dot
            if recording and int(time.time() * 2) % 2 == 0:
                cv2.circle(final, (30, ui_h + 30), 10, (0, 0, 255), -1)

            cv2.imshow("Thermal Scanner", final)

            # ---- Video write ----
            if record_video:
                h, w = final.shape[:2]
                if video_writer is None:
                    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                    fn = os.path.join(output_folder, f"thermal_video_{ts}.avi")
                    video_writer = cv2.VideoWriter(
                        fn, cv2.VideoWriter_fourcc(*'MJPG'), 20.0, (w, h))
                video_writer.write(final)

            # ---- CSV write ----
            if recording:
                now = time.time()
                if now - last_csv_time >= csv_interval:
                    if ENABLE_TRACKER and tracker_pos is not None:
                        tcx = int(round(tracker_pos[0]))
                        tcy = int(round(tracker_pos[1]))
                        pixels = []
                        for ry in range(tcy - TRACKER_BOX_H, tcy + TRACKER_BOX_H + 1):
                            for rx in range(tcx - TRACKER_BOX_W, tcx + TRACKER_BOX_W + 1):
                                if 0 <= ry < temps.shape[0] and 0 <= rx < temps.shape[1]:
                                    pixels.append(temps[ry, rx])
                                else:
                                    pixels.append(np.nan)
                    else:
                        taper  = ANGLE_CORRECTION.get('taper', 1.0)
                        pixels = extract_roi_pixels(temps, RECT_CONFIG, taper=taper, step=step)
                        tcx, tcy = RECT_CONFIG['cx'], RECT_CONFIG['cy']

                    row = [f"{now:.3f}".replace('.', ','),
                           f"{now - csv_start:.3f}".replace('.', ','),
                           f"{tcx}", f"{tcy}"]
                    for val in pixels:
                        row.append(
                            f"{val:.3f}".replace('.', ',') if np.isfinite(val) else "")
                    csv_rows.append(row)
                    csv_frame_count += 1
                    last_csv_time = now

                    # Flush buffer to disk periodically
                    if csv_frame_count % CSV_FLUSH_INTERVAL == 0:
                        csv_writer.writerows(csv_rows)
                        csv_file.flush()
                        csv_rows = []

            # ---- Keyboard ----
            full_key = cv2.waitKeyEx(1)
            if full_key != -1:
                char_key = full_key & 0xFF

                # Arrow keys
                if full_key in [2490368, 0x260000, 65362]:        # UP
                    if is_shift_pressed(): RECT_CONFIG['h'] -= 2
                    else:                  RECT_CONFIG['cy'] -= 2
                elif full_key in [2621440, 0x280000, 65364]:      # DOWN
                    if is_shift_pressed(): RECT_CONFIG['h'] += 2
                    else:                  RECT_CONFIG['cy'] += 2
                elif full_key in [2424832, 0x250000, 65361]:      # LEFT
                    if is_shift_pressed(): RECT_CONFIG['w'] -= 2
                    else:                  RECT_CONFIG['cx'] -= 2
                elif full_key in [2555904, 0x270000, 65363]:      # RIGHT
                    if is_shift_pressed(): RECT_CONFIG['w'] += 2
                    else:                  RECT_CONFIG['cx'] += 2

                # Quit
                elif char_key in (ord('q'), 27):
                    break

                # Offset
                elif char_key in (ord('+'), 43): OFFSET += 0.1
                elif char_key in (ord('-'), 45): OFFSET -= 0.1

                # Taper
                elif char_key in (ord('0'), 48): ANGLE_CORRECTION['taper'] += 0.05
                elif char_key in (ord('9'), 57): ANGLE_CORRECTION['taper'] -= 0.05

                # Start recording
                elif char_key == ord('r'):
                    if not recording:
                        # Open file immediately and write header
                        ts_fn = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                        csv_path = os.path.join(output_folder, f"thermal_data_{ts_fn}.csv")
                        csv_file = open(csv_path, 'w', newline='')
                        csv_writer = csv.writer(csv_file, delimiter=';')
                        # Write header
                        box_h = TRACKER_BOX_H if ENABLE_TRACKER else RECT_CONFIG['h'] // step
                        box_w = TRACKER_BOX_W if ENABLE_TRACKER else RECT_CONFIG['w'] // step
                        hdr = ['timestamp', 'elapsed_s', 'nozzle_cx', 'nozzle_cy']
                        for dr in range(-box_h, box_h + 1):
                            for dc in range(-box_w, box_w + 1):
                                hdr.append(f'px_dr{dr:+d}_dc{dc:+d}')
                        csv_writer.writerow(hdr)
                        csv_file.flush()
                        csv_rows = []; csv_frame_count = 0
                        recording = True; csv_start = time.time()
                        print(f'CSV recording → {csv_path}')
                elif char_key == ord('v'):
                    if not record_video:
                        record_video = True; vid_start = time.time()

                # Manual save — always available, cancels auto-timer for that stream
                elif char_key == ord('t'):
                    if recording:
                        recording = False; csv_start = None
                        if csv_rows: csv_writer.writerows(csv_rows); csv_rows = []
                        csv_file.close(); csv_file = None; csv_writer = None
                        print(f'CSV saved → {csv_path}  ({csv_frame_count} frames)')
                elif char_key == ord('s'):
                    if record_video:
                        record_video = False; vid_start = None
                        if video_writer: video_writer.release(); video_writer = None

    finally:
        cap.release()
        if video_writer: video_writer.release()
        if csv_file:
            if csv_rows: csv_writer.writerows(csv_rows)
            csv_file.close()
            print(f'CSV closed on exit → {csv_path}  ({csv_frame_count} frames)')
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
